# Vehicle Resale Price Regression Analysis
Predict used car resale prices in Indian Rupees (₹) using regression models.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

%matplotlib inline
sns.set_theme(style='darkgrid')

## 1. Load Data & Exploratory Analysis

In [2]:
data_path = Path('data/cars.csv')
df = pd.read_csv(data_path)
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Data Cleaning & Feature Engineering

In [3]:
# Drop duplicates and null values
df = df.drop_duplicates().dropna()
df = df[df['owner'] != 'Test Drive Car']
df = df[(df['selling_price'] > 10000) & (df['km_driven'] > 0)]

# IQR outlier removal (k=3)
for col in ['selling_price', 'km_driven']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    df = df[(df[col] >= q1 - 3*iqr) & (df[col] <= q3 + 3*iqr)]

# Feature engineering: brand and car_age
df['brand'] = df['name'].astype(str).apply(lambda x: x.strip().split()[0].title())
brand_counts = df['brand'].value_counts()
frequent = brand_counts[brand_counts >= 20].index
df['brand'] = df['brand'].apply(lambda b: b if b in frequent else 'Other')
df['car_age'] = 2026 - df['year']
df[['brand', 'car_age', 'km_driven', 'fuel', 'transmission', 'owner', 'selling_price']].head()

## 3. Modeling & Evaluation

In [4]:
numeric_features = ['car_age', 'km_driven']
categorical_features = ['brand', 'fuel', 'transmission', 'owner']
feature_cols = ['brand', 'car_age', 'km_driven', 'fuel', 'transmission', 'owner']

X = df[feature_cols]
y = df['selling_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

model = TransformedTargetRegressor(
    regressor=Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ]),
    func=np.log1p,
    inverse_func=np.expm1
)
model.fit(X_train, y_train)
preds = model.predict(X_test)
print(f'R2: {r2_score(y_test, preds):.4f}')
print(f'MAE: ₹{mean_absolute_error(y_test, preds):,.0f}')